In [2]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import json, os, time, re, sys
from random import uniform


# Cambio por url de pagina 1 para despues ir cambiando a cada pagina

BASE = "https://books.toscrape.com/"
url = urljoin(BASE, "catalogue/page-1.html")
next_page = url



rating_number = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
items = []



# 🔹 ADICIÓN (función): resolver autores por título con Open Library
# Explicación:
# - Open Library es una API pública (sin API key) que permite buscar libros por título.
# - Usamos el endpoint /search.json con el parámetro "title".
# - Traemos hasta 1 mejor resultado y devolvemos la lista de autores (si los hay).
# - Incluye manejo de errores y timeouts para no romper el scraping si la API falla.
OPENLIB_SEARCH = "https://openlibrary.org/search.json"
GOOGLE_BOOKS = "https://www.googleapis.com/books/v1/volumes"

import requests

def get_authors(title, category, lang="en", timeout=5, max_authors=5):
    """
    Devuelve una LISTA de autores usando Open Library y como fallback Google Books.
    Siempre retorna al menos ["Desconocido"] si no encuentra nada.
    """
    def _norm(name: str) -> str:
        return re.sub(r"\s+", " ", name).strip()

    autores = []

    # 1) Open Library (puede traer múltiples)
    try:
        ol = requests.get(
            "https://openlibrary.org/search.json",
            params={"title": title, "limit": 3},
            timeout=timeout
        )
        ol.raise_for_status()
        docs = ol.json().get("docs", [])
        for d in docs:
            for a in d.get("author_name", []) or []:
                a = _norm(a)
                if a and a not in autores:
                    autores.append(a)
                    if len(autores) >= max_authors:
                        break
            if len(autores) >= max_authors:
                break
    except requests.RequestException as e:
        print(f"[⚠️] OpenLibrary error: {e}")

    # 2) Google Books (fallback, agrega si faltan)
    if len(autores) < max_authors:
        query = f'intitle:"{title}"'
        if category:
            query += f' subject:"{category}"'
        try:
            gb = requests.get(
                "https://www.googleapis.com/books/v1/volumes",
                params={
                    "q": query,
                    "maxResults": 3,
                    "orderBy": "relevance",
                    "langRestrict": lang
                },
                timeout=timeout
            )
            gb.raise_for_status()
            items = gb.json().get("items", []) or []
            for it in items:
                for a in it.get("volumeInfo", {}).get("authors", []) or []:
                    a = _norm(a)
                    if a and a not in autores:
                        autores.append(a)
                        if len(autores) >= max_authors:
                            break
                if len(autores) >= max_authors:
                    break
        except requests.RequestException as e:
            print(f"[⚠️] Google Books error: {e}")

    return autores if autores else ["Desconocido"]



# 🔹 ADICIÓN: función para obtener sopa con reintentos
def get_soup(url_pagina_soup):
    for intento in range(3):  # 🔹 ADICIÓN: reintentos
        try:
            resp = requests.get(url_pagina_soup, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException:
            time.sleep(1.2 * (intento + 1))
    raise RuntimeError(f"No pude obtener {url_pagina_soup}")





# mientras existan paginas siguientes, veridicado por boton next. otra forma de hacerlo es iterando
while next_page:

    soup = get_soup(next_page)
    books = soup.find_all("article", class_="product_pod")

    for b in books:


        # Title
        title = b.h3.a["title"]


        # Price
        # 💥💥💥 
        txtprice = b.find("p", class_="price_color").get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", txtprice))


        # URL
        href = b.h3.a["href"]
        book_url = urljoin(next_page, href)


        # detalle 
        soup_detail = get_soup(book_url)


        # category
        breadcrumb_link = soup_detail.select("ul.breadcrumb li a")
        category = breadcrumb_link[-1].get_text(strip=True) if len(breadcrumb_link) >= 3 else "Unknown"


        # Rating                   
        rating_tag = soup_detail.select_one("p.star-rating")
        classes = rating_tag.get("class", []) if rating_tag else []                        
        rating_word = next((c for c in classes if c in rating_number), "One")
        rating = rating_number.get(rating_word, 1)                         


        # Stock
        available = soup_detail.find("p", class_="instock availability").text.strip()
        if available == "In stock":
            in_stock = True
        else: 
            in_stock = False


        # description
        description_tag = soup_detail.find("div", id="product_description")
        description = (
            description_tag.find_next_sibling('p').text.strip()
            if description_tag else None
        )


        # 🔹 ADICIÓN (llamar API): buscar autores por título
        # Explicación:
        # - Usamos la función get_autor(title, category).
        # - Retorna una lista [] con los nombres de los autores o Desconocido.
        # - Se agrega como campo "author" en el item para persistirlo luego en JSON.
        authors = get_authors(title, category)
        # Pequeña pausa de cortesía para no saturar la API si hay muchos libros
        time.sleep(0.15)

        items.append({
            "title": title,
            "category": category,
            "rating": rating,
            "URL": book_url,
            "price": price,
            "stock": in_stock,
            "description": description,
            # 🔹 ADICIÓN (nuevo campo en el dataset): autores como lista
            # Explicación:
            # - Este campo nuevo te permitirá luego crear tablas 'autores' y 'libro_autor' (M:N) en tu DB.
            # - Mantenerlo como lista te conserva todos los coautores que reporte la API.
            "authors": authors
        })

    time.sleep(1)

    # para cada pagina del 1 al 5
    botton_next = soup.find("li", class_="next")
    if botton_next:
        href_next = botton_next.a["href"]
        next_page = urljoin(next_page, href_next)
        time.sleep(0.5)
    else:
        break


# Guardar en JSON
with open('libros_scrapeados.json', 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# Leer desde JSON
with open('libros_scrapeados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
    print(f'Se guardaron {len(datos)} libros')
    print(datos[0])  # Mostrar el primero para validar estructura

# Abrir automáticamente el archivo (solo en Windows)
os.startfile('libros_scrapeados.json')


KeyboardInterrupt: 

In [1]:

import sqlite3

# Creamos (o abrimos) la base de datos
conn = sqlite3.connect("libros.db")

print("✅ Base de datos creada o abierta correctamente")

# Cerramos la conexión (importante)
conn.close()



✅ Base de datos creada o abierta correctamente


In [3]:
import sqlite3

# 1. Conectamos con la base
conn = sqlite3.connect("libros.db")





# 2. Activamos las llaves foráneas (para relaciones entre tablas)
# PRAGMA es una directiva especial de SQLite para configurar opciones
# Activamos las llaves foráneas (para relaciones entre tablas)
conn.execute("PRAGMA foreign_keys = ON;")




# 3. Escribimos el DDL (definición de tablas)
# DDL significa Data Definition Language, o sea: “lenguaje para definir datos”. 
# (Es el conjunto de comandos CREATE, ALTER, DROP, etc.)

# Lo que hay dentro son instrucciones SQL separadas por punto y coma ; 
# Cada CREATE TABLE crea una tabla.
DDL = """
CREATE TABLE IF NOT EXISTS categories (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    description TEXT,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);
"""

# 4. Ejecutamos todas las sentencias
conn.executescript(DDL)

print("✅ Tablas creadas correctamente")

# 5. Cerramos la conexión
conn.close()


✅ Tablas creadas correctamente


In [ ]:



PRAGMA foreign_keys = ON;



-- Definición de tablas para la base de datos de libros
-- crea, una tabla, si no existe, con el nombre 'categories'
CREATE TABLE IF NOT EXISTS categories (
    -- Definición de columnas
    -- id es la clave primaria (PK)
    -- name es texto obligatorio y único
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);


-- crea, una tabla, si no existe, con el nombre 'authors'
CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);


-- crea, una tabla, si no existe, con el nombre 'books'
CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    -- REAL para números con decimales
    price       REAL NOT NULL,
    -- check es una restricción para validar datos
    -- rating es un entero entre 1 y 5
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    -- TEXT para descripción larga
    description TEXT,
    category_id INTEGER NOT NULL,
    -- ON DELETE sirve para definir el comportamiento al borrar
    -- Restrict evita borrar una categoría si hay libros asociados
    -- llave foranea, referencia a categories(id), referencia a, columna id de tabla categories, ON DELETE RESTRICT
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);


-- crea, una tabla, si no existe, con el nombre 'book_author'
CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,


    --
    -- Solamente es necesario si el usuario ingresa datos manualmente
    --

    -- Foreign Keys (FK) es una restricción para validar datos
    -- Definimos la clave primaria compuesta (PK)
    -- sirve para evitar duplicados en la tabla muchos a muchos (M:N)
    PRIMARY KEY (book_id, author_id),

    -- Foreign Keys (FK) sirve para definir que hacer al borrar
    -- ON DELETE CASCADE borra en cascada los registros relacionados 
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);

